# Exercise: Auditing a Real Model for Bias, Fairness and Compliance
## AIAT — Generative AI | Unit 4: Ethics & Regulations

## Learning Objectives

After completing this exercise, you will be able to:
- Measure a **real, documented** disparity in a real dataset and its impact on model fairness
- Apply a fairness-aware training technique (dropping the protected attribute + sample reweighting)
- Implement transparency tools (feature importance, prediction explanations)
- Audit a model against a GDPR / EU AI Act compliance checklist
- Produce a structured AI Ethics Report that reports trade-offs honestly

## Real-World Context

You are a responsible AI engineer. Your team has trained a survival-prediction model on the
**real Titanic passenger manifest** — 891 real people, a real life-or-death outcome, and a
**disparity that is historical fact**: 74.2% of the women aboard survived versus 18.9% of the men.

**Read that sentence again, because it is the point of this exercise.** That gap was *not* written
into the data by this notebook. Nothing here injects a group penalty and then congratulates you
for finding it. Your job is the real job: diagnose how the disparity flows into a model, apply a
fix, **measure honestly what the fix cost**, and document all of it for the compliance team.

The procedure is identical to the one a bank runs on a loan model or a hospital on a triage
model. Only the dataset is one where you can verify the ground truth for yourself.

**Theory → Practice:** Review the examples in `../examples/` first:
- `01_generative_ai_ethics.ipynb` — ethics survey + the fairness-audit workflow
- `04_applying_ai_regulatory_guidelines_gdpr.ipynb` — GDPR applied to this same real manifest

---


## 📥 Inputs & 📤 Outputs

**Inputs:** `load("titanic")` — the real Titanic manifest (891 real passengers; `Age` has 177
genuine missing values). The whole file ships with the repository, so the loader finds it from
any working directory and downloads it on Google Colab. No synthetic data anywhere in this
exercise.

**Outputs:** Fairness metrics before/after your fix, a feature-importance chart, a GDPR
compliance table, and an Ethics Report.

**Expected when complete — and read this before you assume what "success" looks like:**
with the mitigation prescribed in Task 2, the demographic-parity gap should shrink **a lot**, but on this real data it will
**not** drop under the 10% flag, and **overall accuracy will fall by roughly ten points**. That is
not you doing the exercise wrong. Reporting a mitigation that helped one metric while costing
another *is* the deliverable — a report claiming the fix simply "worked" would be false.


In [ ]:
# Setup: load the REAL Titanic passenger manifest.
# WHY real data: a synthetic "biased loan dataset" hands your audit back exactly the disparity
# its author typed in - the discovery is circular and teaches nothing. Here the disparity is a
# documented historical fact, so every number you compute below is evidence, not an echo.
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# ── Load the real manifest ─────────────────────────────────────────────────
# --- Data setup. Works from any folder, and on Google Colab. -------------------------
# WHAT: find the repository root and put it on sys.path, then import the shared loader.
# WHY:  a hard-coded '../../../Course 04/datasets/raw/...' only resolves when the kernel's
#       working directory happens to be this notebook's folder. This does not care.
import sys, pathlib

_here = pathlib.Path.cwd().resolve()
_root = next((p for p in [_here, *_here.parents] if (p / "tools" / "data.py").exists()), None)
if _root is None:                     # Google Colab, or a stray copy of the notebook
    import urllib.request
    pathlib.Path("tools").mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/A-Alwabel/AI-Diploma-Program/main/tools/data.py", "tools/data.py")
    except Exception as _e:
        raise RuntimeError(
            "Could not find the AI Diploma repository from this folder, and could not "
            "download the data loader either. Open this notebook inside a clone of "
            "https://github.com/A-Alwabel/AI-Diploma-Program, or connect to the internet "
            f"and re-run this cell. (underlying error: {_e})") from None
    _root = pathlib.Path.cwd()
sys.path.insert(0, str(_root))

from tools.data import load
# -------------------------------------------------------------------------------------
df = load("titanic")
print(f'Loaded {len(df)} real passenger records.')

# Real data is incomplete. Report the gaps, then impute the one column we model with,
# and say what we did - silent imputation is itself an ethics failure.
print(f"  genuine missing ages: {int(df['Age'].isna().sum())}")
age_median = df['Age'].median()
df['Age'] = df['Age'].fillna(age_median)
print(f'  imputed with the median age ({age_median:.1f})')

# ── The protected attribute ────────────────────────────────────────────────
# group = 1 for female (314 passengers), 0 for male (577). NOTE the trap: the SMALLER group
# here is the historically FAVOURED one. Real data does not follow the textbook template in
# which "minority" and "disadvantaged" are the same word - check, never assume.
group = (df['Sex'] == 'female').astype(int).values

# ── Assemble features (group included on purpose - you will audit it) ─────
# group is at index 3 so you can drop it cleanly in Task 2.
feature_names = ['Pclass', 'Age', 'Fare', 'group']
X = np.column_stack([df['Pclass'], df['Age'], df['Fare'], group]).astype(float)
y = df['Survived'].values

# ── Split, keeping group labels aligned with the train/test rows ──────────
X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(
    X, y, group, test_size=0.3, random_state=42, stratify=y
)

print('\nDataset ready.')
print(f'  Training samples : {len(X_train)}')
print(f'  Test samples     : {len(X_test)}')
print(f'  Group sizes (train): male={int((g_train == 0).sum())}, female={int((g_train == 1).sum())}')
print('\nHISTORICAL survival rate in the test set (fact, not injected by this notebook):')
print(f'  group 0 (male,   the larger group) : {y_test[g_test == 0].mean():.2%}')
print(f'  group 1 (female, the smaller group): {y_test[g_test == 1].mean():.2%}')
print('\nThat gap is what your model is about to learn. Your task is not to be shocked by it -')
print('it is to measure what the model does with it, and what your fix costs.')


---
## Task 1: Train a Baseline Model and Measure the Disparity (25 points)

Train a Logistic Regression classifier on all features **including** `group`.  
Then compute the **demographic parity difference** (the gap in *predicted* positive rate
between the two groups).

**Demographic Parity Difference** = |P(predict survive | group=0) − P(predict survive | group=1)|

A value > 0.10 is typically flagged for investigation.

⚠️ **A measured gap is not automatically an injustice.** "Women and children first" was an
announced policy in 1912, so here a large gap is the model faithfully reproducing a stated rule.
The metric tells you outcomes differ; only human judgment decides whether the difference is
justified. Keep that distinction — you will need it in the Ethics Report.


In [ ]:
# TODO 1a: Train a LogisticRegression model on X_train, y_train (all 4 features, group included)
# Hint: LogisticRegression(max_iter=1000)
model_biased = None  # <-- replace with your code

# TODO 1b: Predict on X_test
y_pred_biased = None  # <-- replace with your code

# TODO 1c: Compute overall accuracy
acc_biased = None  # <-- replace with your code

# TODO 1d: Compute the PREDICTED survival rate for each group separately
# (majority = group 0 = male, the larger group; minority = group 1 = female, the smaller one)
rate_majority = None  # P(predict=1 | g_test == 0)
rate_minority = None  # P(predict=1 | g_test == 1)

# TODO 1e: Compute the demographic parity difference
dp_diff = None  # |rate_majority - rate_minority|

print(f'Accuracy         : {acc_biased:.2%}')
print(f'Predicted survive (group 0, male)  : {rate_majority:.2%}')
print(f'Predicted survive (group 1, female): {rate_minority:.2%}')
print(f'Demographic Parity Difference: {dp_diff:.2%}')
if dp_diff and dp_diff > 0.10:
    print('⚠️  LARGE DISPARITY — gap exceeds the 10% flag; it must be investigated, not ignored.')
    print('    Compare it against the historical rates printed in the setup cell before you')
    print('    write a single word about what it means.')


---
## Task 2: Apply a Fairness Fix — Blinding + Sample Reweighting (25 points)

One common fix is to **remove the protected attribute** from the features and apply **sample
weights** that upweight the smaller group during training, forcing the model to give it equal
influence.

Steps:
1. Drop the `group` column from X (index 3)
2. Compute per-sample weights: group 1 gets weight = (n_group0 / n_group1), group 0 gets 1.0
3. Retrain with the `sample_weight` parameter
4. Measure how much the demographic parity difference changed — **and what else changed**

⚠️ **Do not assume this "works".** On real data this fix does two things at once. Record both:
- the parity gap and how far it moved (does it clear the 10% flag?)
- **overall accuracy, and accuracy for each group separately** — a fix that buys parity by
  becoming wrong about everybody is not a fix, it is a worse model with a better metric.

Add the per-group accuracy computation yourself; the TODOs below only cover the parity numbers.


In [ ]:
# TODO 2a: Drop the group column from train and test sets
X_train_fair = None  # X_train without column index 3
X_test_fair  = None  # X_test  without column index 3

# TODO 2b: Compute sample weights
# n_group1 = number of training samples where g_train == 1 (female)
# n_group0 = number of training samples where g_train == 0 (male)
# weight for group 1 = n_group0 / n_group1
# weight for group 0 = 1.0
sample_weights = None  # numpy array of shape (len(X_train),)

# TODO 2c: Train a new LogisticRegression with sample_weight
model_fair = None  # <-- your code

# TODO 2d: Predict and compute metrics
y_pred_fair = None
acc_fair = None
rate_majority_fair = None
rate_minority_fair = None
dp_diff_fair = None

# TODO 2e: also compute accuracy SEPARATELY for each group, before and after the fix.
# The single accuracy number can stay flat while one group's accuracy collapses.
acc_group0_fair = None   # accuracy on g_test == 0
acc_group1_fair = None   # accuracy on g_test == 1

print('=== After Fairness Fix ===')
print(f'Accuracy         : {acc_fair:.2%}')
print(f'Predicted survive (group 0, male)  : {rate_majority_fair:.2%}')
print(f'Predicted survive (group 1, female): {rate_minority_fair:.2%}')
print(f'Demographic Parity Difference: {dp_diff_fair:.2%}')
print(f'\nParity gap  : {dp_diff:.2%} → {dp_diff_fair:.2%}')
print(f'Accuracy    : {acc_biased:.2%} → {acc_fair:.2%}')
print(f'Per-group accuracy after the fix: group 0 {acc_group0_fair:.2%}, group 1 {acc_group1_fair:.2%}')
print('\nWrite down BOTH movements. If the gap shrank but is still above 10%, say so.')
print('If accuracy fell, say by how much and who paid for it. That is the report.')


---
## Task 3: Transparency — Feature Importance (20 points)

EU law constrains solely automated decisions, but read the wording carefully — the word
"explanation" never appears in **GDPR Article 22**. Article 22(3) gives the data subject the
right "to obtain human intervention on the part of the controller, to express his or her point
of view and to contest the decision". The transparency duty sits in **Articles 13–15**, which
require *"meaningful information about the logic involved"* (Art. 15(1)(h)). A right to an
explanation *of one specific decision* appears only in **Recital 71**, which is interpretive and
not binding — which is exactly why Wachter, Mittelstadt & Floridi (2017) titled their paper
*Why a Right to Explanation of Automated Decision-Making Does Not Exist in the General Data
Protection Regulation*.

Producing "meaningful information about the logic involved" is the engineering task here.
Use the coefficients from your fair model to visualize which features drive the decision.

Then write 2–3 sentences interpreting the chart. In particular: with `group` removed, which
feature carries the most weight — and could that feature be acting as a **proxy** for the
attribute you just deleted?


In [ ]:
# TODO 3a: Extract coefficients from model_fair
# model_fair.coef_[0]  →  array of shape (3,) since we dropped the group column
fair_feature_names = ['Pclass', 'Age', 'Fare']
coefs = None  # <-- your code

# TODO 3b: Plot a horizontal bar chart of feature importance (absolute value of coefs)
# Use plt.barh() and label each bar with the feature name
# Title: 'Feature Importance (Fair Model)'

# YOUR PLOT CODE HERE

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=80)
plt.close()
print('Chart saved.')

# TODO 3c: Print your interpretation
# Which feature has the highest importance? Is that expected?
# Ticket class and fare correlate with who got a lifeboat place. If the blinded model leans on
# them, has the protected attribute really been removed - or only its column name?


---
## Task 4: GDPR / EU AI Act Compliance Audit (15 points)

High-risk AI systems (like loan or triage decisions) must satisfy several legal requirements.  
Complete the audit checklist below by filling in **True/False** and a one-line **evidence** note.

Be strict with yourself. Every `satisfied` value must point at something you actually did in
this notebook. "We removed the group column" is evidence for *unawareness*, which — as Task 3
should have shown you — is **not** the same as evidence for non-discrimination.


In [ ]:
# TODO 4: Fill in the compliance checklist
# For each item, set satisfied = True or False, and write a short evidence string

checklist = [
    {
        'requirement': 'GDPR Art.22 — Human oversight of automated decisions',
        'satisfied': None,    # True / False
        'evidence': '???'     # e.g. 'A human reviews every negative prediction before it is used'
    },
    {
        'requirement': 'GDPR Art.22 — Right to explanation for the data subject',
        'satisfied': None,
        'evidence': '???'     # e.g. 'Per-decision coefficient contributions available (Task 3)'
    },
    {
        'requirement': 'EU AI Act Art.9 — Bias testing before deployment',
        'satisfied': None,
        'evidence': '???'     # e.g. 'Demographic parity measured before and after mitigation'
    },
    {
        'requirement': 'EU AI Act Art.13 — Transparency documentation',
        'satisfied': None,
        'evidence': '???'     # e.g. 'Model card records the real dataset, the 177 imputed ages,'
                              #      'and the accuracy cost of the mitigation'
    },
    {
        'requirement': 'Protected attribute removed from features (unawareness)',
        'satisfied': None,
        'evidence': '???'     # e.g. 'group column dropped in Task 2 — but state honestly whether'
                              #      'the residual parity gap shows proxies still carry the signal'
    },
    {
        'requirement': 'Residual disparity measured and disclosed after mitigation',
        'satisfied': None,
        'evidence': '???'     # This is the one a checklist usually forgets. Do not.
    },
]

# ── Print the audit table: every item needs a True/False + evidence line ──
print('=' * 60)
print('AI ETHICS COMPLIANCE AUDIT')
print('=' * 60)
for item in checklist:
    status = '✅' if item['satisfied'] else '❌'
    print(f"{status} {item['requirement']}")
    print(f"   Evidence: {item['evidence']}")
    print()


---
## Task 5: Ethics Report (15 points)

Write a short Ethics Report (3–5 sentences per section) in the cell below.  
This mirrors what a responsible AI team would submit to a regulator.

### Your Ethics Report

**1. Problem Statement**  
*(What disparity is in this data? How did you measure it? State explicitly that it is a
historical fact of the 1912 record and not an artifact of the code.)*  
> TODO: Write here

**2. Root Cause Analysis**  
*(Why does the disparity exist — the evacuation policy, the confounding with ticket class, the
label itself? Which of these is a data problem and which is a history problem?)*  
> TODO: Write here

**3. Mitigation Applied**  
*(What did you do? Give the parity gap before and after AND the accuracy before and after,
including per group. Did the gap clear the 10% flag? Say plainly if it did not.)*  
> TODO: Write here

**4. Residual Risk**  
*(The model is not fair now. What remains? Consider proxy features, the accuracy you spent,
which group paid for it, and the fact that "disparity removed" is not the same as
"decision justified".)*  
> TODO: Write here

**5. Monitoring Plan**  
*(How would you detect the disparity returning in production? Which metrics, at what cadence,
and what threshold triggers a human review?)*  
> TODO: Write here


---
## 📝 Summary

In this exercise you:
- Measured **demographic parity** on a **real, historically documented** disparity — not one a
  script injected and then rediscovered
- Applied **blinding + sample reweighting** and measured what the mitigation **cost**, not only
  what it improved
- Used **model coefficients** to explain predictions, and asked whether the remaining features
  are **proxies** for the attribute you deleted
- Audited the system against **GDPR / EU AI Act** requirements
- Produced a structured **Ethics Report** — a real deliverable in industry

**Key takeaways:**
1. Fairness is not automatic — it must be actively measured, mitigated, and documented.
2. Mitigations trade one metric against another. A report that shows only the improved metric
   is advocacy, not an audit.
3. Deleting a protected column does not delete the correlation: class and fare still carry it.
4. A measured disparity is evidence that outcomes differ. Whether that difference is *unjust* is
   a human judgment about context — which is exactly the judgment a synthetic dataset with an
   injected penalty quietly takes away from you.

---
## 📚 References

- [GDPR Article 22 — Automated Decision Making](https://gdpr-info.eu/art-22-gdpr/) and [Article 15(1)(h)](https://gdpr-info.eu/art-15-gdpr/) — "meaningful information about the logic involved"
- Wachter, S., Mittelstadt, B. & Floridi, L. (2017). *Why a Right to Explanation of Automated Decision-Making Does Not Exist in the General Data Protection Regulation*. International Data Privacy Law 7(2), 76–99. https://academic.oup.com/idpl/article/7/2/76/3860948
- [EU AI Act (2024)](https://artificialintelligenceact.eu/)
- Hardt, M., Price, E. & Srebro, N. (2016). *Equality of Opportunity in Supervised Learning*. NeurIPS. https://arxiv.org/abs/1610.02413
- Dwork, C. et al. (2012). *Fairness Through Awareness* — why "unawareness" fails. ITCS. https://arxiv.org/abs/1104.3913
- Barocas, S., Hardt, M. & Narayanan, A. (2023). *Fairness and Machine Learning*. MIT Press. https://fairmlbook.org/
- [Fairlearn — Microsoft Fairness Toolkit](https://fairlearn.org/)
- [Google ML Fairness](https://developers.google.com/machine-learning/fairness-overview)
